<a href="https://colab.research.google.com/github/Mc-cloud/chessRL/blob/main/agents/Agent_NN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install chess

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 55.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for chess: filename=chess-1.11.2-py3-none-any.whl size=147775 sha256=81a943f4c8b90141fbf446ed66998f4db9f27460764c50c2b2ea210083668bb5
  Stored in directory: /root/.cache/pip/wheels/83/1f/4e/8f4300f7dd554eb8de70ddfed96e94d3d030ace10c5b53d447
Successfully built chess


In [ ]:
from utils_NN import (
    UCI_TO_IDX, IDX_TO_UCI, move_to_index, index_to_move,
        board_to_tensor, Node, MCTS, CNN,
        play_one_game, generate_games_with_self_play,
)

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

class ChessDataset(Dataset):
    def __init__(self, dataset_total):
        self.dataset = dataset_total

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        state, policy, value = self.dataset[idx]
        return (
            state.clone().detach(),
            torch.tensor(policy, dtype=torch.float32),
            torch.tensor([value], dtype=torch.float32)
        )

def alpha_zero_loss(log_policy_preds, value_preds, policy_targets, value_targets):
  """
  Combine l'erreur sur le score (Value) et l'erreur sur les coups (Policy).
  """
  value_loss = F.mse_loss(value_preds, value_targets)

  policy_loss = -torch.sum(policy_targets * log_policy_preds, dim=1).mean()

  return value_loss + policy_loss

def train_network(neural_net, dataset_total, epochs=10, batch_size=64, learning_rate=0.001):
    """
    Prend le réseau actuel et l'entraîne sur les données du Self-Play.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    neural_net.to(device)

    optimizer = optim.Adam(neural_net.parameters(), lr=learning_rate, weight_decay=1e-4)

    dataset = ChessDataset(dataset_total)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    neural_net.train()

    print(f"Début de l'entraînement sur {device} avec {len(dataset_total)} positions...")

    for epoch in range(epochs):
        total_loss = 0.0

        for states, policies, values in dataloader:
            states = states.to(device)
            policies = policies.to(device)
            values = values.to(device)

            optimizer.zero_grad()

            log_policy_preds, value_preds = neural_net(states)

            loss = alpha_zero_loss(log_policy_preds, value_preds, policies, values)

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs} - Loss moyenne : {total_loss / len(dataloader):.4f}")

    neural_net.to("cpu")
    print("Entraînement terminé !")



In [ ]:
import chess

def evaluate_new_net(old_net, new_net, num_games=20, mcts_simulations=100, win_threshold=0.55):
    """
    Fait s'affronter l'ancien et le nouveau réseau.
    Retourne True si le nouveau réseau est significativement meilleur.
    """
    print(f"\n⚔️ Bienvenue dans l'Arène ! Début du match en {num_games} parties... ⚔️")
    new_wins = 0
    old_wins = 0
    draws = 0

    old_net.eval()
    new_net.eval()

    for i in range(num_games):
        board = chess.Board()

        # on alterne les couleurs.
        if i % 2 == 0:
            white_net = new_net
            black_net = old_net
            new_is_white = True
        else:
            white_net = old_net
            black_net = new_net
            new_is_white = False

        # On instancie des MCTS tout neufs pour vider leur mémoire entre chaque partie
        white_mcts = MCTS(white_net, n_simulations=mcts_simulations)
        black_mcts = MCTS(black_net, n_simulations=mcts_simulations)

        while not board.is_game_over():
            if board.turn == chess.WHITE:
                policy_dict = white_mcts.search(board)
            else:
                policy_dict = black_mcts.search(board)

            best_move = max(policy_dict, key=policy_dict.get)
            board.push(chess.Move.from_uci(best_move))

        result = board.result()
        if result == "1-0":
            if new_is_white: new_wins += 1
            else: old_wins += 1
        elif result == "0-1":
            if not new_is_white: new_wins += 1
            else: old_wins += 1
        else:
            draws += 1

        print(f"Partie {i+1}/{num_games} terminée | Score global -> Nouveau: {new_wins} | Ancien: {old_wins} | Nuls: {draws}")

    total_score = new_wins + (0.5 * draws)
    win_rate = total_score / num_games

    print(f"\n📊 Ratio de victoire du Challenger : {win_rate:.1%}")

    if win_rate >= win_threshold:
        print("👑 Succès ! Le Nouveau Réseau a surpassé le maître. Il devient le standard.")
        return True
    else:
        print("❌ Échec. Le Nouveau Réseau est rejeté. On garde l'Ancien pour la prochaine génération.")
        return False

In [ ]:
import copy

best_network = CNN(input_channels=13, board_size=8, action_size=len(UCI_TO_IDX))
iteration = 1
while True:
    print(f"=== GÉNÉRATION {iteration} ===")
    dataset = generate_games_with_self_play(best_network, num_games=1000)

    challenger_network = copy.deepcopy(best_network)

    train_network(challenger_network, dataset)
    is_better = evaluate_new_net(old_net=best_network, new_net=challenger_network)

    if is_better:
        best_network = challenger_network

    iteration += 1

=== GÉNÉRATION 1 ===

🔄 Début de la génération de 1000 parties en Self-Play...
♟️ Partie 1/1000 en cours...
♟️ Partie 2/1000 en cours...
♟️ Partie 3/1000 en cours...
